<div align="center">
<img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
<h1>Practice 10: TensorFlow for Manufacturing Sound Classification</h1>
</div>

ME 5995: AI in Mechanical Engineering I, Wayne State University  
Optional, ungraded self-study. Student version.  
Estimated core route: 60 minutes. Optional review: 30 minutes.

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1)  
[Practice Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Practice_index.ipynb)

[Open in Colab](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/practice/week10/practice10_tensorflow_sound_classification.ipynb)

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Overview and learning objectives

Practice 9 explained how sound becomes a log-mel input. Here you use TensorFlow's
Keras interface, `tf.keras`, to construct, train and interpret a small classifier.
Follow a complete 32-neuron example, then reconstruct the same APIs in a short
24-neuron rehearsal. The rehearsal uses only 64 training inputs and two epochs
to practice coding; it is not a model-quality or architecture comparison.

By the end, you can trace tensor/batch shapes, build a Sequential model, connect
integer labels to softmax and sparse cross-entropy, use compile/fit/predict,
read training history, restore validation-selected weights and interpret errors.
You will also practice runtime help and explicit shape inspection needed in HW5
and the coding exam. There are six small code tasks and five core short responses.

Run guided code unchanged, attempt each TRY IT and consult the self-check afterward.
Empty task cells are comments so guided examples run before you attempt them.
The final rehearsal tasks build on one another and must be completed in order.
An empty task is not an answer. No Canvas submission or accuracy target applies.

## Data and engineering context

Use the same 800 normal MIMII recordings as Lab 8/9 and HW5, ten one-second inputs
per recording. Each input is 32 mel bands by 61 time frames. The four integer
labels are 0 fan, 1 pump, 2 slider and 3 valve. This is a course-derived machine-type
classification activity, not the original anomaly-detection benchmark or fault
diagnosis. The source is MIMII public 1.0, Channel 1, 16 kHz PCM16, 0 dB background
mixture. Log-mel uses a numerical power reference, not calibrated dB SPL.

The supplied manifest keeps all segments from a parent together: 480/160/160
training/validation/test recordings, or 4800/1600/1600 inputs. IDs represented in
training also appear in evaluation, so unseen-machine generalization is not tested.
IDs identify parents; neither IDs nor labels are input features. Practice 9's
smaller public waveform pool has a different split and fitted scaling statistics.

## Required files and provided setup

Use TensorFlow (including tf.keras), NumPy, pandas, Matplotlib, scikit-learn and
IPython. For local work, install the required packages in your local Python
environment. If requirements.txt is provided, it is an optional installation aid.
No package installation cell is added to Colab here.

The provided setup uses the `mimii_logmel800` cache and retrieves missing course
files from the course GitHub location. No checksum code or private validation
assertions are part of student work. Restart and run top to bottom.

In [ ]:
# PROVIDED: reproducible framework setup, not a student task.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display
tf.get_logger().setLevel("ERROR")
tf.config.experimental.enable_op_determinism()
print("TensorFlow version:", tf.__version__)
print("Keras version:", tf.keras.__version__)

In [ ]:
# PROVIDED: use cached files, or retrieve the supplied course data.
data_base_url = "https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/mimii_logmel800/"
data_folder = Path("mimii_logmel800")
data_folder.mkdir(parents=True, exist_ok=True)
feature_file = data_folder / "logmel.npz"
metadata_file = data_folder / "recordings.csv"

In [ ]:
# PROVIDED: download only missing files. Do not edit this setup cell.
from urllib.request import urlretrieve
if not feature_file.is_file():
    urlretrieve(data_base_url + "logmel.npz", feature_file)
if not metadata_file.is_file():
    urlretrieve(data_base_url + "recordings.csv", metadata_file)
reference_file = data_folder / "baseline32_summary.csv"
if not reference_file.is_file():
    urlretrieve(data_base_url + "baseline32_summary.csv", reference_file)
dictionary_file = data_folder / "FEATURE_DICTIONARY.md"
if not dictionary_file.is_file():
    urlretrieve(data_base_url + "FEATURE_DICTIONARY.md", dictionary_file)
license_file = data_folder / "LICENSE_DATA.txt"
if not license_file.is_file():
    urlretrieve(data_base_url + "LICENSE_DATA.txt", license_file)

## 1. Inspect the data before using TensorFlow

NPZ stores named NumPy arrays. A CSV stores a table. Read their structure first.
One `X` row is one segment; `recording_index` links it to a parent row in the
recording table. A shape `(8000,)` is a one-dimensional vector, not 8000 columns.

In [ ]:
archive = np.load(feature_file)
recordings = pd.read_csv(metadata_file)
print("NPZ keys:", archive.files)
X = archive["X"]
y = archive["y"]
recording_index = archive["recording_index"]
recording_ids = archive["recording_id"]
class_names = archive["class_names"]
print("X shape/dtype:", X.shape, X.dtype)
print("y shape/dtype:", y.shape, y.dtype)
print("Recording table shape:", recordings.shape)
display(recordings[["recording_id", "machine_type", "machine_id", "partition"]].head())

In [ ]:
example_parent = recording_index[0]
print("One input matrix:", X[0].shape)
print("First input's parent row:", example_parent)
print("First input's parent ID:", recording_ids[example_parent])
print("First input's target:", y[0], class_names[y[0]])
print("First twelve parent rows:", recording_index[:12])

### Reuse the taught split and training-global scaling

This is a concise repeat of Practice 9's principle. Assign parent partitions to
segments and fit one scalar mean/SD on all training elements. Use the same pair
for validation/test. No waveform extraction or split generation is needed here.

In [ ]:
recording_partitions = recordings["partition"].to_numpy()
segment_partitions = recording_partitions[recording_index]
train_mask = segment_partitions == "train"
validation_mask = segment_partitions == "validation"
test_mask = segment_partitions == "test"
X_train = X[train_mask]
y_train = y[train_mask]
X_validation = X[validation_mask]
y_validation = y[validation_mask]
X_test = X[test_mask]
y_test = y[test_mask]

In [ ]:
training_mean = X_train.mean()
training_std = X_train.std()
X_train_scaled = (X_train - training_mean) / training_std
X_validation_scaled = (X_validation - training_mean) / training_std
X_test_scaled = (X_test - training_mean) / training_std
print("Training statistics:", training_mean, training_std)
print("Training:", X_train_scaled.shape, y_train.shape)
print("Validation:", X_validation_scaled.shape, y_validation.shape)
print("Test:", X_test_scaled.shape, y_test.shape)

### A scalar has shape `()`

`X_train.mean()` and `X_train.std()` pool all training elements because no axis
is specified. Each result is one scalar, with shape `()`: there are no array axes.
This is different from `(1,)`, a one-dimensional array containing one value.
Reusing these scalars changes the values, not the shape `(n,32,61)` of a partition.
The leading count n is the number of segments in that partition, not recordings.

In [ ]:
# PROVIDED: inspect statistics, not a new normalization.
print("Mean shape:", training_mean.shape)
print("Standard-deviation shape:", training_std.shape)
print("Validation before/after:", X_validation.shape, X_validation_scaled.shape)

## 2. Read batch shapes and a forward pass

A tensor is a numerical array used by TensorFlow. `tf.convert_to_tensor` changes
the container; it does not train a model or normalize values. Keras also accepts
NumPy arrays directly in fit/predict, as the course examples do.

The first axis counts inputs in a batch. Each input retains 32 mel bands and
61 time frames. `layer(input_tensor)` computes a forward pass through a layer;
it is different from calling `fit`, which updates learned weights.

In [ ]:
batch_array = X_train_scaled[:8]
batch_tensor = tf.convert_to_tensor(batch_array, dtype=tf.float32)
print("NumPy batch:", batch_array.shape, batch_array.dtype)
print("Tensor batch:", batch_tensor.shape, batch_tensor.dtype)
flatten_layer = tf.keras.layers.Flatten()
flat_batch = flatten_layer(batch_tensor)
print("After Flatten:", flat_batch.shape)

In [ ]:
tf.keras.utils.set_random_seed(9)
hidden_layer = tf.keras.layers.Dense(32, activation="relu")
output_layer = tf.keras.layers.Dense(4, activation="softmax")
hidden_batch = hidden_layer(flat_batch)
output_batch = output_layer(hidden_batch)
print("Hidden outputs:", hidden_batch.shape)
print("Class probabilities:", output_batch.shape)
print("Hidden weight matrix:", hidden_layer.kernel.shape)
print("Hidden bias vector:", hidden_layer.bias.shape)

### TRY IT 1: Change the number of inputs, not the representation

Create `practice_batch` from the first 16 scaled training inputs and convert it
to `practice_tensor` with dtype tf.float32. Use the existing flatten_layer to
create `practice_flat`. Print all shapes. No weights need to be fitted.

In [ ]:
# TODO: Write your code here using the completed example.

Why does Flatten change the last dimension to 1952 while preserving batch size? Also report training_mean.shape and briefly explain its meaning. Short phrases are enough.

**Your response:** TODO.

## 3. Construct the complete 32-neuron model

Sequential applies layers in list order. Input declares one input's shape;
the batch dimension is omitted. Flatten combines its two feature axes without
learning weights. Dense(32) learns weights/biases and applies ReLU; Dense(4) converts
its four scores to softmax probabilities. Integer targets still have shape `(n,)`.

The standalone layers above only illustrated dimensions. Build a fresh complete
model below and reset its seed, so those demonstrations do not change the training
example. `None` in model.input_shape means variable batch size, not missing data.

In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(9)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 61)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(4, activation="softmax")
])
model.summary()

In [ ]:
hidden_parameters = 1952 * 32 + 32
output_parameters = 32 * 4 + 4
total_parameters = hidden_parameters + output_parameters
print("Hidden weights and biases:", hidden_parameters)
print("Output weights and biases:", output_parameters)
print("Total:", total_parameters)
print("Model input/output shapes:", model.input_shape, model.output_shape)

## 4. Compile and fit with explicit arguments

`compile` configures learning; it does not train. Adam controls weight updates.
Sparse categorical cross-entropy expects integer class indices such as 0,1,2,3;
we do not need one-hot labels here. Softmax already produces probabilities, so
this course uses the default probability form of the loss, not from_logits=True.
Accuracy is reported as a metric; cross-entropy is the optimized loss.

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### Provided checkpoint setup: run unchanged

One epoch passes through all training inputs. With 4800 inputs and batch size 32, there
are 150 batches per epoch. Validation is measured after epochs and does not update
weights. ModelCheckpoint stores the weights with the smallest validation loss.
It does not stop training; the example still runs 40 epochs. Test is not passed
to fit or checkpoint selection. Saving/restoring infrastructure is supplied.

In [ ]:
checkpoint_folder = Path("practice10_checkpoints")
checkpoint_folder.mkdir(exist_ok=True)
checkpoint_file = checkpoint_folder / "baseline32.weights.h5"
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    checkpoint_file, monitor="val_loss", mode="min",
    save_best_only=True, save_weights_only=True, verbose=0
)

In [ ]:
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_validation_scaled, y_validation),
    epochs=40, batch_size=32, shuffle=True,
    callbacks=[checkpoint], verbose=0
)
print("Completed epochs:", len(history.history["loss"]))

Which inputs update weights and which inputs monitor the checkpoint? How many training batches are in one epoch here?

**Your response:** TODO.

## 5. Read history and restore the selected weights

Keras returns a History object. Its `history` item maps metric names to lists;
reading the supplied keys does not require constructing a Python dictionary.
We retrieve each list explicitly and plot epochs starting at 1.

In [ ]:
training_losses = history.history["loss"]
validation_losses = history.history["val_loss"]
training_accuracies = history.history["accuracy"]
validation_accuracies = history.history["val_accuracy"]
epoch_numbers = np.arange(1, 41)

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(epoch_numbers, training_losses, label="Training")
plt.plot(epoch_numbers, validation_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Sparse categorical cross-entropy")
plt.title("One fixed 32-neuron training run")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(epoch_numbers, training_accuracies, label="Training")
plt.plot(epoch_numbers, validation_accuracies, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Segment accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_position = np.argmin(validation_losses)
best_epoch = best_position + 1
best_validation_loss = validation_losses[best_position]
print("Selected epoch:", best_epoch)
print("Minimum validation loss:", best_validation_loss)
print("Final training/validation loss:", training_losses[-1], validation_losses[-1])
# PROVIDED: use the saved weights, rather than retraining a different model.
model.load_weights(checkpoint_file)

Does a low final training loss prove good generalization? Use the plotted validation history to justify why the selected epoch can differ from epoch 40.

**Your response:** TODO.

## 6. Predict probabilities, then obtain class indices

`predict` does not update weights. One row contains four class probabilities in
the class_names order. Sum along axis1 to check each row; use argmax along axis1
to select a class for each input. Taking argmax without an axis would give one
index for the whole array. A probability is model confidence, not proof of correctness.

### Small completed example: choose within each row

Each row below represents one segment, and each column represents one class in
the usual fan/pump/slider/valve order. No training is involved. Read the rows
yourself before running the code: the predicted indices should be 0, 1 and 3.
Axis 1 is the class axis; selecting along it leaves one label for each input row.
The number of labels follows the number of rows, not the number of classes.

In [ ]:
# GUIDED: three segments and four class probabilities per segment.
example_probabilities = np.array([
    [0.6, 0.2, 0.1, 0.1],
    [0.1, 0.5, 0.3, 0.1],
    [0.1, 0.1, 0.2, 0.6]
])
example_predictions = np.argmax(example_probabilities, axis=1)
print("Probability shape:", example_probabilities.shape)
print("Predicted indices:", example_predictions)
print("Predicted-label shape:", example_predictions.shape)
print("First probability-row sum:", example_probabilities[0].sum())

In [ ]:
validation_probabilities = model.predict(X_validation_scaled, verbose=0)
validation_predictions = np.argmax(validation_probabilities, axis=1)
validation_accuracy = accuracy_score(y_validation, validation_predictions)
print("Probability shape:", validation_probabilities.shape)
print("Predicted-label shape:", validation_predictions.shape)
print("First probability row:", validation_probabilities[0])
print("First row sum:", validation_probabilities[0].sum())
print("First predicted/true type:", class_names[validation_predictions[0]], class_names[y_validation[0]])
print("Selected-checkpoint validation accuracy:", validation_accuracy)

## 7. Evaluate the frozen model and distinguish observation units

The model and checkpoint rule were fixed before this test call. Report the result;
do not tune after inspecting test. This reused course cohort is not an untouched
external benchmark. Recording grouping prevents sibling leakage, but does not
establish unseen-ID or acquisition-session independence.

In [ ]:
test_probabilities = model.predict(X_test_scaled, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)
test_segment_accuracy = accuracy_score(y_test, test_predictions)
print("Test segment probability/label shapes:", test_probabilities.shape, test_predictions.shape)
print("Test segment accuracy:", test_segment_accuracy)

### Provided recording aggregation: run unchanged

Inputs retain consecutive groups of ten per parent. Reshape uses that guaranteed
ordering to average **probability vectors**, then take argmax. It does not average
nominal class indices. Each parent contributes one recording prediction, while
segment accuracy counts each one-second input separately.

In [ ]:
number_of_test_recordings = len(y_test) // 10
grouped_probabilities = test_probabilities.reshape(number_of_test_recordings, 10, 4)
recording_probabilities = grouped_probabilities.mean(axis=1)
recording_predictions = np.argmax(recording_probabilities, axis=1)
grouped_true_labels = y_test.reshape(number_of_test_recordings, 10)
recording_true_labels = grouped_true_labels[:, 0]
test_recording_accuracy = accuracy_score(recording_true_labels, recording_predictions)
recording_confusion = confusion_matrix(recording_true_labels, recording_predictions, labels=[0, 1, 2, 3])
print("Recording probabilities:", recording_probabilities.shape)
print("Test recording accuracy:", test_recording_accuracy)

In [ ]:
confusion_display = ConfusionMatrixDisplay(recording_confusion, display_labels=class_names)
confusion_display.plot(cmap="Blues", colorbar=False)
plt.xlabel("Predicted machine type")
plt.ylabel("True machine type")
plt.title("Selected checkpoint: test recordings")
plt.tight_layout()
plt.show()

Read `recording_confusion[row,column]` as a count of **actual row-class**
recordings assigned to the **predicted column-class**. Indices start at zero and
follow class_names. For example, [0,1] means actual fan, predicted pump. It counts
recordings, not segments or percentages. A zero means no such error in this run.

Report the validation probability and predicted-label shapes. Then interpret recording_confusion[0,1] using its actual value and explain why recording and segment accuracy can differ.

**Your response:** TODO.

## 8. Use runtime help for a specific question

The exam uses Colab. Runtime `help()` for Python and installed official packages
is allowed; external searches, prior course materials and AI assistance are not.
Help printed inside Colab is not permission to follow external links. Practice
the relevant APIs now rather than discovering an unfamiliar API during the exam.

Ask a narrow question: for example, what does `axis` mean in argmax? Pass the
function object to help, without executing it inside the help call.

In [ ]:
help(np.argmax)

Find the **axis** parameter and read what happens when it is omitted.
For training arguments, type `help(model.fit)` in a new code cell and locate
`validation_data`, `epochs`, `batch_size` and `callbacks`. Do not read every option.
The commented line below avoids saving a long fit documentation dump during Run all.
The argument names used in this notebook are also summarized in the next section.

In [ ]:
# Optional lookup when needed: uncomment the following line in Colab.
# help(model.fit)

## 9. Reconstruct the taught workflow in small steps

This is an API rehearsal, not a model comparison. Use a hidden layer with 24 ReLU units,
four softmax outputs, Adam with learning rate 0.001, sparse categorical cross-entropy and accuracy.
Fit for **two epochs**, batch size 16, on the supplied 64 training inputs and monitor
the supplied 16 validation inputs. The small sets span the ordered training and
validation arrays so they include all four classes. They do not contain test data.
Use the same already-fitted training-global scaling; do not fit scaling again.

Complete TRY IT2 through TRY IT6 in order. No plotting, checkpoint or recording
aggregation implementation is required. The full 32-neuron example above shows
each operation separately. The smaller rehearsal is not evidence that 24 units
are better or worse than the HW model. It is not the complete HW5 solution.

In [ ]:
# PROVIDED: small, fixed subsets only for practicing the APIs.
rehearsal_train_rows = np.linspace(0, len(y_train) - 1, 64, dtype=int)
rehearsal_validation_rows = np.linspace(0, len(y_validation) - 1, 16, dtype=int)
X_rehearsal_train = X_train_scaled[rehearsal_train_rows]
y_rehearsal_train = y_train[rehearsal_train_rows]
X_rehearsal_validation = X_validation_scaled[rehearsal_validation_rows]
y_rehearsal_validation = y_validation[rehearsal_validation_rows]
tf.keras.utils.set_random_seed(19)
print("Rehearsal training:", X_rehearsal_train.shape, y_rehearsal_train.shape)
print("Rehearsal validation:", X_rehearsal_validation.shape, y_rehearsal_validation.shape)

### TRY IT 2: Construct the model and inspect its size

Create `practice_model` with Input(shape=(32,61)), Flatten, Dense(24) with ReLU and
Dense(4) with softmax. Print its input/output shapes and call summary. Calculate the
parameter count as `(1952*24+24) + (24*4+4)` and compare with the summary.

In [ ]:
# TODO: Write your code here using the completed example.

### TRY IT 3: Configure learning

Create a **new** `practice_optimizer` using Adam with learning_rate=0.001.
Compile practice_model using sparse_categorical_crossentropy and metrics=["accuracy"].
Construct the optimizer and compile the model in separate statements.

In [ ]:
# TODO: Write your code here using the completed example.

### TRY IT 4: Train with the correct validation pair

Create `practice_history` by fitting practice_model on X_rehearsal_train and
y_rehearsal_train. Supply validation_data=(X_rehearsal_validation,y_rehearsal_validation),
epochs=2, batch_size=16, shuffle=True and verbose=0. No checkpoint is needed for
this tiny API-only run. Report the number of history entries, which should be 2.

In [ ]:
# TODO: Write your code here using the completed example.

### TRY IT 5: Predict first, then choose classes

Predict on X_rehearsal_validation into `practice_probabilities`. Convert it to
`practice_predictions` with np.argmax(...,axis=1). Print probability/label shapes
and the first probability-row sum. Use separate statements, not nested calls.

In [ ]:
# TODO: Write your code here using the completed example.

### TRY IT 6: Score the matching targets

Use accuracy_score(y_rehearsal_validation,practice_predictions) to create
`practice_accuracy`. Print it and the first predicted/true machine-type names.
The output checks that your code runs. No particular rehearsal accuracy is required.

In [ ]:
# TODO: Write your code here using the completed example.

## 10. Diagnose a short workflow error

These are discussion snippets, not cells to execute. All corrections use APIs
already demonstrated. An error message is useful evidence; do not guess until
you have checked the array shapes and required target/loss combination.

| Snippet | Check |
|---|---|
| Input(shape=(1952,)) followed by raw 32x61 matrices | Does the declared shape match one supplied input? |
| Dense(3, activation="softmax") | How many machine-type labels are possible? |
| loss="categorical_crossentropy" with integer targets | Does the loss expect integer labels or one-hot targets? |
| training_mean = X.mean() | Which partitions influenced fitted preprocessing? |
| np.argmax(example_probabilities, axis=0), producing shape (4,) | The required shape is (3,). Which axis compares classes within a segment? |
| Choose the epoch with the best test score | Which partition may select a checkpoint? |

Correct the loss-name and argmax snippets, then briefly explain why X.mean()
is not the fitting rule. For the argmax correction, use example_probabilities
and give one short reason for your axis choice. The wrong shape (4,) and desired
shape (3,) are provided; do not derive them or explain the wrong indices.
Write the corrected expression in your response; no extra executable cell is needed.

**Your response:** TODO.

## Optional challenge: Explain the code without copying it

After completing the core route, close the self-check and use only this notebook's
stated settings plus runtime help to reconstruct a fresh 32-unit **untrained**
model, compile statement and the text of a fit call. Do not execute another full
training run or change test-selected settings. Explain the purpose of each
argument, the expected probability shape and the two Dense parameter counts.
This adds no graded work or new model search.

## Completion checklist

No Canvas submission is required for this optional Practice. Save a personal copy.
Rendered Markdown checkboxes are not clickable; edit `[ ]` to `[x]`.

- [ ] I inspected feature/label/parent structure before training.
- [ ] I traced batch, Flatten, hidden and output shapes.
- [ ] I can explain compile, fit and predict as separate operations.
- [ ] I attempted all six code tasks and five short responses.
- [ ] I can distinguish final weights from a validation-selected checkpoint.
- [ ] I used runtime help to answer a specific API question.
- [ ] I can interpret probability axes and segment/recording evaluation.
- [ ] I compared with the self-check after attempting the rehearsal.

## References and attribution

- [Keras Sequential guide](https://keras.io/guides/sequential_model/).
- [Keras training APIs: compile, fit, predict](https://keras.io/api/models/model_training_apis/).
- [Keras Dense layer](https://keras.io/api/layers/core_layers/dense/).
- [Keras ModelCheckpoint](https://keras.io/api/callbacks/model_checkpoint/).
- [Python runtime help](https://docs.python.org/3/library/functions.html#help).
- [MIMII public 1.0](https://zenodo.org/records/3384388), DOI: 10.5281/zenodo.3384388,
  Purohit et al. (2019), [dataset paper](https://arxiv.org/abs/1909.09347).

Original data authors: Harsh Purohit, Ryo Tanabe, Kenji Ichige, Takashi Endo,
Yuki Nikaido, Kaori Suefusa and Yohei Kawaguchi (Hitachi, Ltd.). Course changes:
selected normal 0 dB recordings, retained Channel 1, segmented ten-second parents,
computed fixed-reference log-mel and added labels/lineage/partition metadata.
Read the supplied FEATURE_DICTIONARY.md and LICENSE_DATA.txt; preserve
[CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) attribution and
ShareAlike requirements. This course task is distinct from the original benchmark.
Online reference links support ordinary study; exam access is restricted to
the examination notebook and approved runtime help.